# Training on soft labels: typed-decisions

> **⚠️ Plan for a long run.** The LoRA and QLoRA sections train the backbone on the full training
> data, and nothing can be cached when the backbone learns, so each one takes far longer than
> head-only training. As a rough guide, Qwen3-0.6B takes about an hour per epoch on a Mac, so up
> to about two hours per section with the defaults, and a fraction of that on an A100. Use a CUDA
> GPU (on Colab, an L4 or A100 runtime) and save trained models to Google Drive with
> `model.save(...)`, since Colab deletes `/content` when the runtime disconnects. A run stopped
> with Ctrl-C keeps its best weights so far and continues from where it stopped when `fit` is
> called again with the same data.

[LocalLLaMA/typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) has 1,200
training and 400 test records from four operational workflows: agent-trace monitoring, customer
service, invoice processing and security incidents. Each record is a structured JSON state with the
same five questions for its workflow.

Its labels are **soft**: each answer is a probability spread from several raters, for example
`{"continue": 0.28, "human_review": 0.43, "observe": 0.25, "stop": 0.03}`. sysone trains on these
distributions directly (passed as `gold`) and evaluates against them, so accuracy here means "picks
the raters' top answer", and **distance** shows how close the predicted spread is.

It trains Qwen3-0.6B with `tuning="head"`, then fine-tunes the backbone with `tuning="lora"` and
`tuning="qlora"`. [`encoder_typed_decisions.ipynb`](encoder_typed_decisions.ipynb) is the same
notebook with ModernBERT, a bidirectional encoder.

In [ ]:
# On Google Colab, or anywhere sysone isn't installed yet, install it from PyPI.
import importlib.util

if importlib.util.find_spec("sysone") is None:
    %pip install -q "sysone>=0.2"

In [ ]:
import json

from datasets import load_dataset

from sysone import DecoderModel, baselines, compare

BACKBONE_EPOCHS = 2  # upper bound for LoRA and QLoRA; early stopping usually ends sooner

## Load the data

`state`, `questions` and `gold` are JSON text, which sysone decodes itself. The workflow is used as
the row's `category`.

In [ ]:
def load_rows(split):
    ds = load_dataset("LocalLLaMA/typed-decisions", "all", split=split)
    return [{"state": r["state"], "questions": r["questions"], "gold": r["gold"], "category": r["workflow"]}
            for r in ds]


train = load_rows("train")
test = load_rows("test")
print("%d training records, %d test records" % (len(train), len(test)))
print("rater probabilities for one question:", json.loads(train[0]["gold"])["action"]["probabilities"])

## Before training: baselines

- **random guess**: every option equally likely.
- **average answer**: for each workflow question, the raters' average distribution in the
  training data. Every record in a workflow has the same questions, so this is a strong bar: a
  model that beats it is reading each record, not just learning the typical answer.

In [ ]:
results = baselines(test, train=train)
compare(results)

## Train

The soft `gold` distributions are the training targets. 10% of records are held back for
validation and early stopping, split evenly across workflows.

In [ ]:
model = DecoderModel(tuning="head")
model.fit(train)

## After training

In [ ]:
results["head"] = model.evaluate(test)
compare(results)

## Fine-tuning the backbone with LoRA

With `tuning="lora"`, small adapters on the top 60% of the kept layers train along with the head,
so the backbone itself adapts to these records. Nothing can be cached, since the backbone's
outputs change every step: each example costs a forward and a backward pass through the backbone
every epoch. It trains on the same records as the head-only model above, so the two compare
directly.

Validation runs about ten times per epoch, and training stops early once it stops improving, so
`BACKBONE_EPOCHS` is an upper bound.

In [ ]:
lora = DecoderModel(tuning="lora", epochs=BACKBONE_EPOCHS)
lora.fit(train)
results["lora"] = lora.evaluate(test)
compare(results)

## Fine-tuning the backbone with QLoRA

`tuning="qlora"` (the default) trains the same adapters as LoRA, but stores the backbone's frozen
weights in 4-bit, so it needs much less GPU memory. It is somewhat slower per step, because the
weights are unpacked on the fly, and accuracy is usually close to LoRA's.

4-bit loading needs bitsandbytes and an NVIDIA GPU (any Colab GPU runtime). Elsewhere, for
example on a Mac, QLoRA falls back to LoRA with a warning and trains exactly like the section
above, so skip this section there. The cell prints which mode it actually used.

In [ ]:
qlora = DecoderModel(tuning="qlora", epochs=BACKBONE_EPOCHS)
qlora.fit(train)
print("tuning used:", qlora.tuning_)  # "lora" means 4-bit wasn't available and it fell back
memory = qlora.backbone_.get_memory_footprint() / 1e9, lora.backbone_.get_memory_footprint() / 1e9
print("backbone memory: %.2f GB with QLoRA, %.2f GB with LoRA" % memory)
results["qlora"] = qlora.evaluate(test)
compare(results)

**Metrics** (from `model.evaluate`, the same for hard and soft labels):
- **accuracy**: the model's most likely option is the right one (for soft labels: the raters'
  top answer). For a score question this means the exact level.
- **distance**: how far the predicted probabilities are from the label (0 = identical, 1 = completely
  different).
- **log_loss**: cross-entropy against the label; lower is better.
- **score_error**: for score questions, the gap between predicted and true expected level.
- **auc**: for yes/no questions, how well the model ranks yes above no (0.5 = chance).

With soft labels, **distance** and **log_loss** matter as much as accuracy: when raters split
45/55, either answer is defensible, but the probabilities should reflect the split.

In [ ]:
compare(results, by="overall")

### By workflow

In [ ]:
compare(results, by="category")

## Reading the results

- Raters often disagree, so accuracy has a ceiling well below 100%.
- **average answer** is the bar to beat. With a frozen backbone, the head has limited room to get
  past it; fine-tuning the backbone with LoRA or QLoRA is what should move these numbers further.
- QLoRA should land close to LoRA; a gap of a point or two is within run-to-run noise.
- Probabilities are not calibrated yet (temperatures stay at 1.0), which affects **distance** and
  **log_loss** more than accuracy.
- A long run stopped with Ctrl-C continues from where it stopped with `lora.fit(train)`.